In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
import re
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.preprocessing import MinMaxScaler


In [2]:
nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /Users/isabel/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/isabel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
INPUT_DIR = 'fromGoogleDrive'

In [4]:
# preprocessing (same as compcor)
def preprocess(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"^\s*-\s*", "", text) # Remove dashes at the beginning of texts.
    text = re.sub(r"^\s*\d+\.\s*", "", text) # Remove numbers in 1., 2., 3. format at the beginning of the text.
    words = word_tokenize(text) # tokenize
    return words

In [5]:
# Get dataset-specific information.
rows = []
for file in os.listdir(f'./{INPUT_DIR}/datasets/datasetsPrep/'):

    # Process texts (lowercase, remove punctuation and numbers).
    list_of_texts = [preprocess(t) for t in pd.read_csv(f'./{INPUT_DIR}/datasets/datasetsPrep/{file}')['text'].dropna().tolist()]
    # Flatten words.
    all_words = [word for doc in list_of_texts for word in doc]
    total_words = len(all_words)
    freq = Counter(all_words)
    # Filter stopwords.
    filtered_words = [w for w in all_words if w not in stop_words]
    filtered_word_total = len(filtered_words)
    stopword_ratio = (total_words - filtered_word_total) / total_words if total_words > 0 else 0
    # Average word length.
    avg_word_len = np.mean([len(w) for w in all_words]) if all_words else 0
    # Get document lengths.
    doc_lengths = np.array([len(t) for t in list_of_texts])
    # Vocab.
    vocab = set(all_words)
    vocab_len = len(vocab)
    # Type-Token Ratio.
    ttr = vocab_len / total_words if total_words > 0 else 0
    # Rare words (hapax legomena).
    hapax = sum(1 for _, c in freq.items() if c == 1)

    # Rare words (dis legomena).
    dis = sum(1 for _, c in freq.items() if c == 2)

    # Top-N Coverage (Frequency Concentration)
    top_10_count = sum(c for _, c in freq.most_common(10))
    coverage = top_10_count / total_words if total_words > 0 else 0
    # Append everything to a row. 
    rows.append({
    "Dataset": file.replace('.csv', ''),
    "Total Documents": len(list_of_texts),
    "Total Words": total_words,
    "Total Words (without stopwords)": filtered_word_total,
    "Total Stop Words": total_words - filtered_word_total,
    "Stopword Ratio": stopword_ratio,
    "Average Word Length": avg_word_len,
    "Vocab Length": vocab_len,
    "Type-Token Ratio": ttr,
    "Hapax Legomena": hapax,
    "Dis Legomena": dis,
    "Top 10 Words Coverage": coverage,
    "Mean Document Length": doc_lengths.mean(),
    "Median Document Length": np.median(doc_lengths),
    "Standard Deviation Document Length": doc_lengths.std(),
    "Min Document Length": doc_lengths.min(),
    "Max Document Length": doc_lengths.max(),
    "25th Document Length Percentile": np.percentile(doc_lengths, 25),
    "75th Document Length Percentile": np.percentile(doc_lengths, 75),
    })

# Make a dataframe.
dataset_characteristics_df = pd.DataFrame(rows)
# Remove non-comparable metrics (e.g. number of documents) for easy visualization. 
dataset_characteristics_df.drop(columns=['Total Documents', 'Total Words', 'Total Words (without stopwords)', 'Total Stop Words', 'Vocab Length', 'Hapax Legomena', 'Dis Legomena', 'Standard Deviation Document Length', 'Min Document Length', 'Max Document Length', '25th Document Length Percentile', '75th Document Length Percentile'])

,Dataset,Stopword Ratio,Average Word Length,Type-Token Ratio,Top 10 Words Coverage,Mean Document Length,Median Document Length
0,atis,0.425348,4.707788,0.015657,0.352524,11.367818,11.0
1,banking77,0.379056,3.584884,0.017522,0.310065,13.249216,11.0
2,clinc150,0.498641,3.826311,0.036851,0.283618,8.525612,8.0
3,clinicalDialogueSummarizations,0.343715,4.360843,0.035360,0.267681,46.278934,17.0
4,dementiaAudio,0.438171,3.480664,0.026194,0.386441,116.338798,105.0
5,huffPostNews,0.343072,4.199670,0.023536,0.238591,25.163032,23.0
6,medicalAbstracts,0.286589,5.109807,0.021201,0.263213,205.660064,200.0
7,simSUM,0.101771,4.106722,0.012611,0.381137,104.821400,103.0
8,syntheticCareHomeNurseNotes,0.318930,4.937623,0.027334,0.277734,26.532596,24.0
9,yahoo,0.387502,3.922505,0.029172,0.255604,47.845493,42.0


In [6]:
# Get dataset-specific information.
rows = []
for file in os.listdir(f'./{INPUT_DIR}/datasets/datasetsPrep/'):
    if 'dementia' not in file:

        # Process texts (lowercase, remove punctuation and numbers).
        list_of_texts = [preprocess(t) for t in pd.read_csv(f'./{INPUT_DIR}/datasets/datasetsPrep/{file}')['text'].dropna().tolist()]
        # Flatten words.
        all_words = [word for doc in list_of_texts for word in doc]
        total_words = len(all_words)
        freq = Counter(all_words)
        # Filter stopwords.
        filtered_words = [w for w in all_words if w not in stop_words]
        filtered_word_total = len(filtered_words)
        stopword_ratio = (total_words - filtered_word_total) / total_words if total_words > 0 else 0
        # Average word length.
        avg_word_len = np.mean([len(w) for w in all_words]) if all_words else 0
        # Get document lengths.
        doc_lengths = np.array([len(t) for t in list_of_texts])
        # Vocab.
        vocab = set(all_words)
        vocab_len = len(vocab)
        # Type-Token Ratio.
        ttr = vocab_len / total_words if total_words > 0 else 0
        # Rare words (hapax legomena).
        hapax = sum(1 for _, c in freq.items() if c == 1)

        # Rare words (dis legomena).
        dis = sum(1 for _, c in freq.items() if c == 2)

        # Top-N Coverage (Frequency Concentration)
        top_10_count = sum(c for _, c in freq.most_common(10))
        coverage = top_10_count / total_words if total_words > 0 else 0
        # Append everything to a row. 
        rows.append({
        "Dataset": file.replace('.csv', ''),
        "Total Documents": len(list_of_texts),
        "Total Words": total_words,
        "Total Words (without stopwords)": filtered_word_total,
        "Total Stop Words": total_words - filtered_word_total,
        "Stopword Ratio": stopword_ratio,
        "Average Word Length": avg_word_len,
        "Vocab Length": vocab_len,
        "Type-Token Ratio": ttr,
        "Hapax Legomena": hapax,
        "Dis Legomena": dis,
        "Top 10 Words Coverage": coverage,
        "Mean Document Length": doc_lengths.mean(),
        "Median Document Length": np.median(doc_lengths),
        "Standard Deviation Document Length": doc_lengths.std(),
        "Min Document Length": doc_lengths.min(),
        "Max Document Length": doc_lengths.max(),
        "25th Document Length Percentile": np.percentile(doc_lengths, 25),
        "75th Document Length Percentile": np.percentile(doc_lengths, 75),
        })

# Make a dataframe.
dataset_characteristics_df = pd.DataFrame(rows)
# Remove non-comparable metrics (e.g. number of documents) for easy visualization. 
dataset_characteristics_df.drop(columns=['Total Documents', 'Total Words', 'Total Words (without stopwords)', 'Total Stop Words', 'Vocab Length', 'Hapax Legomena', 'Dis Legomena', 'Standard Deviation Document Length', 'Min Document Length', 'Max Document Length', '25th Document Length Percentile', '75th Document Length Percentile'])

,Dataset,Stopword Ratio,Average Word Length,Type-Token Ratio,Top 10 Words Coverage,Mean Document Length,Median Document Length
0,atis,0.425348,4.707788,0.015657,0.352524,11.367818,11.0
1,banking77,0.379056,3.584884,0.017522,0.310065,13.249216,11.0
2,clinc150,0.498641,3.826311,0.036851,0.283618,8.525612,8.0
3,clinicalDialogueSummarizations,0.343715,4.360843,0.035360,0.267681,46.278934,17.0
4,huffPostNews,0.343072,4.199670,0.023536,0.238591,25.163032,23.0
5,medicalAbstracts,0.286589,5.109807,0.021201,0.263213,205.660064,200.0
6,simSUM,0.101771,4.106722,0.012611,0.381137,104.821400,103.0
7,syntheticCareHomeNurseNotes,0.318930,4.937623,0.027334,0.277734,26.532596,24.0
8,yahoo,0.387502,3.922505,0.029172,0.255604,47.845493,42.0


In [7]:
def make_output_metric_ranking(df):
    normalized_df = df.copy()
    normalized_df = normalized_df
    metric_cols = [
        "Accuracy",
        "Weighted Accuracy",
        "Time",
        "Monotonicity",
        "Separability",
        "Linearity"
    ]

    for col in metric_cols:
        min_val = normalized_df[col].min()
        max_val = normalized_df[col].max()
        normalized_df[col] = (normalized_df[col] - min_val) / (max_val - min_val)

    winners = {f"{metric}": [] for metric in metric_cols}
    for dataset in list(set(normalized_df['dataset1'])):
            # print(f"---------------- {dataset} ----------------")
            temp_df = normalized_df[(normalized_df['dataset1'] == dataset) | (normalized_df['dataset2'] == dataset)]
            temp_df['Overall'] = temp_df[metric_cols].mean(axis=1)
            temp_df.groupby('metric')[metric_cols].mean()
            grouped_df = temp_df.groupby('metric')[metric_cols].mean()
            grouped_df['Algorithm'] = grouped_df.index
            for metric in metric_cols:
                format_df = pl.from_pandas(grouped_df.sort_values(by=metric, ascending=False))
                winners[metric].append(format_df['Algorithm'][0])

    normalized_df['Overall'] = normalized_df[metric_cols].mean(axis=1)
    sorted_mean_df = normalized_df.groupby('metric')[metric_cols + ['Overall']].mean()
    sorted_mean_df['Algorithm'] = sorted_mean_df.index
    polars_output = pl.from_pandas(sorted_mean_df.sort_values(by='Overall', ascending=False))
    print(polars_output)



    all_winners = []
    for metric, values in winners.items():
        print(metric, Counter(values))
        all_winners.extend(values)

    Counter(all_winners)
    return polars_output

In [8]:
all_temp_dfs = []
for combination in os.listdir(f'./{INPUT_DIR}/outputCompcor/ksc'):
    if os.path.isdir(f'./{INPUT_DIR}/outputCompcor/ksc/{combination}'):
        combination_splits = combination.split('_')
        dataset1, dataset2, repetitions = combination_splits[0], combination_splits[1], combination_splits[3]
        if 'dementia' not in dataset1 and 'dementia' not in dataset2:
            temp_df = pd.read_csv(f'./{INPUT_DIR}/outputCompcor/ksc/{combination}/{combination}_ksc_metrics_measures.csv')
            temp_df = temp_df.groupby('metric').mean().reset_index()
            temp_df['dataset1'] = dataset1
            temp_df['dataset2'] = dataset2
            temp_df['repetitions'] = repetitions
            all_temp_dfs.append(temp_df)

df_ksc = pd.concat(all_temp_dfs)
ksc = make_output_metric_ranking(df_ksc)


shape: (10, 8)
┌──────────┬─────────────┬──────────┬─────────────┬────────────┬───────────┬──────────┬────────────┐
│ Accuracy ┆ Weighted    ┆ Time     ┆ Monotonicit ┆ Separabili ┆ Linearity ┆ Overall  ┆ Algorithm  │
│ ---      ┆ Accuracy    ┆ ---      ┆ y           ┆ ty         ┆ ---       ┆ ---      ┆ ---        │
│ f64      ┆ ---         ┆ f64      ┆ ---         ┆ ---        ┆ f64       ┆ f64      ┆ str        │
│          ┆ f64         ┆          ┆ f64         ┆ f64        ┆           ┆          ┆            │
╞══════════╪═════════════╪══════════╪═════════════╪════════════╪═══════════╪══════════╪════════════╡
│ 0.960912 ┆ 0.938161    ┆ 0.001275 ┆ 0.976853    ┆ 0.910648   ┆ 0.974217  ┆ 0.793678 ┆ MAUVE      │
│ 0.947468 ┆ 0.921543    ┆ 0.000026 ┆ 0.97103     ┆ 0.900664   ┆ 0.975605  ┆ 0.786056 ┆ ZERO       │
│ 0.92304  ┆ 0.890403    ┆ 0.000276 ┆ 0.952817    ┆ 0.849525   ┆ 0.959559  ┆ 0.762603 ┆ TRADITIONA │
│          ┆             ┆          ┆             ┆            ┆           ┆

In [9]:
all_temp_dfs = []
for combination in os.listdir(f'./{INPUT_DIR}/outputCompcor/ksc_synth'):
    if os.path.isdir(f'./{INPUT_DIR}/outputCompcor/ksc_synth/{combination}'):
        combination_splits = combination.split('_')
        dataset1, dataset2, repetitions = combination_splits[0], combination_splits[1], combination_splits[3]
        if 'dementia' not in dataset1 and 'dementia' not in dataset2:
            temp_df = pd.read_csv(f'./{INPUT_DIR}/outputCompcor/ksc_synth/{combination}/{combination}_ksc_metrics_measures.csv')
            temp_df = temp_df.groupby('metric').mean().reset_index()
            temp_df['dataset1'] = dataset1
            temp_df['dataset2'] = dataset2
            temp_df['repetitions'] = repetitions
            all_temp_dfs.append(temp_df)

df_ksc_synth = pd.concat(all_temp_dfs)
ksc_synth = make_output_metric_ranking(df_ksc_synth)


shape: (10, 8)
┌──────────┬─────────────┬──────────┬─────────────┬────────────┬───────────┬──────────┬────────────┐
│ Accuracy ┆ Weighted    ┆ Time     ┆ Monotonicit ┆ Separabili ┆ Linearity ┆ Overall  ┆ Algorithm  │
│ ---      ┆ Accuracy    ┆ ---      ┆ y           ┆ ty         ┆ ---       ┆ ---      ┆ ---        │
│ f64      ┆ ---         ┆ f64      ┆ ---         ┆ ---        ┆ f64       ┆ f64      ┆ str        │
│          ┆ f64         ┆          ┆ f64         ┆ f64        ┆           ┆          ┆            │
╞══════════╪═════════════╪══════════╪═════════════╪════════════╪═══════════╪══════════╪════════════╡
│ 0.85654  ┆ 0.806411    ┆ 0.010739 ┆ 0.927103    ┆ 0.832722   ┆ 0.946736  ┆ 0.730042 ┆ DC         │
│ 0.866206 ┆ 0.815964    ┆ 0.0003   ┆ 0.933658    ┆ 0.769083   ┆ 0.942996  ┆ 0.721368 ┆ TRADITIONA │
│          ┆             ┆          ┆             ┆            ┆           ┆          ┆ L          │
│ 0.736819 ┆ 0.682549    ┆ 0.914143 ┆ 0.784939    ┆ 0.389587   ┆ 0.804098  ┆

In [10]:
df_all = pd.concat([df_ksc, df_ksc_synth])

In [11]:
def make_dataset_dependent_results(input_df):
    metric_cols = [
        "Accuracy",
        "Weighted Accuracy",
        "Time",
        "Monotonicity",
        "Separability",
        "Linearity"
    ]
    dataset_grouped_dfs = []
    for dataset in dataset_characteristics_df['Dataset']:
        temp_df = input_df[
            (input_df['dataset1'] == dataset) | 
            (input_df['dataset2'] == dataset)
        ].copy()
        scaler = MinMaxScaler()
        temp_df[metric_cols] = scaler.fit_transform(temp_df[metric_cols])
        # normalize here
        temp_df['overall'] = temp_df[metric_cols].mean(axis=1)
        dataset_grouped_temp = temp_df.groupby('metric')[metric_cols + ['overall']].mean().sort_values(by='overall', ascending=False)
        dataset_grouped_temp['dataset'] = dataset
        dataset_grouped_dfs.append(dataset_grouped_temp)

    grouped_df = pd.concat(dataset_grouped_dfs)
    return grouped_df

In [12]:
ksc_synth

Accuracy,Weighted Accuracy,Time,Monotonicity,Separability,Linearity,Overall,Algorithm
f64,f64,f64,f64,f64,f64,f64,str
0.85654,0.806411,0.010739,0.927103,0.832722,0.946736,0.730042,"""DC"""
0.866206,0.815964,0.0003,0.933658,0.769083,0.942996,0.721368,"""TRADITIONAL"""
0.736819,0.682549,0.914143,0.784939,0.389587,0.804098,0.718689,"""IRPR"""
0.859017,0.811942,0.00003,0.917858,0.739767,0.928302,0.709486,"""ZERO"""
0.833163,0.775197,0.0013,0.917189,0.790016,0.925602,0.707078,"""MAUVE"""
0.861703,0.810074,0.00197,0.89886,0.681367,0.912584,0.694426,"""FID"""
0.837577,0.799773,0.155006,0.865222,0.5856,0.87055,0.685621,"""ZIPF"""
0.790482,0.726319,0.120917,0.886135,0.641392,0.896279,0.676921,"""CLASSIFIER"""
0.745286,0.687186,0.010741,0.813103,0.60793,0.853342,0.619598,"""PR"""


In [13]:
grouped_ksc = make_dataset_dependent_results(df_ksc)
grouped_ksc_synth = make_dataset_dependent_results(df_ksc_synth)
all_grouped = make_dataset_dependent_results(df_all)

In [14]:
grouped_ksc

,Accuracy,Weighted Accuracy,Time,Monotonicity,Separability,Linearity,overall,dataset
metric,,,,,,,,
MAUVE,0.976293,0.960862,0.001300,0.981445,0.932076,0.976638,0.804769,atis
ZERO,0.953805,0.927357,0.000038,0.979291,0.944219,0.983563,0.798046,atis
FID,0.969957,0.953420,0.001932,0.947740,0.873018,0.958658,0.784121,atis
TRADITIONAL,0.940975,0.909068,0.000356,0.963179,0.882732,0.967957,0.777378,atis
CLASSIFIER,0.867315,0.814681,0.121807,0.924810,0.730216,0.922367,0.730199,atis
...,...,...,...,...,...,...,...,...
DC,0.783164,0.719207,0.010659,0.895872,0.821947,0.905418,0.689378,yahoo
IRPR,0.695967,0.650900,0.905760,0.715674,0.235857,0.730559,0.655786,yahoo
PR,0.707145,0.653417,0.010664,0.756896,0.559702,0.808009,0.582639,yahoo


In [15]:
all_grouped.to_excel('./allDatasetResults.xlsx')

In [16]:
for algorithm in ['MAUVE', 'ZERO']:
    for grouped_name, grouped_df in [('grouped_ksc', grouped_ksc), ('grouped_ksc_synth', grouped_ksc_synth), ('all_grouped', all_grouped)]:
        print(algorithm, grouped_name)
        print(pl.from_pandas(grouped_df[grouped_df.index ==algorithm]))
        


MAUVE grouped_ksc
shape: (9, 8)
┌──────────┬─────────────┬──────────┬─────────────┬────────────┬───────────┬──────────┬────────────┐
│ Accuracy ┆ Weighted    ┆ Time     ┆ Monotonicit ┆ Separabili ┆ Linearity ┆ overall  ┆ dataset    │
│ ---      ┆ Accuracy    ┆ ---      ┆ y           ┆ ty         ┆ ---       ┆ ---      ┆ ---        │
│ f64      ┆ ---         ┆ f64      ┆ ---         ┆ ---        ┆ f64       ┆ f64      ┆ str        │
│          ┆ f64         ┆          ┆ f64         ┆ f64        ┆           ┆          ┆            │
╞══════════╪═════════════╪══════════╪═════════════╪════════════╪═══════════╪══════════╪════════════╡
│ 0.976293 ┆ 0.960862    ┆ 0.0013   ┆ 0.981445    ┆ 0.932076   ┆ 0.976638  ┆ 0.804769 ┆ atis       │
│ 0.966388 ┆ 0.945099    ┆ 0.001295 ┆ 0.981281    ┆ 0.906602   ┆ 0.978251  ┆ 0.796486 ┆ banking77  │
│ 0.946528 ┆ 0.918281    ┆ 0.001258 ┆ 0.974182    ┆ 0.934622   ┆ 0.973422  ┆ 0.791382 ┆ clinc150   │
│ 0.956934 ┆ 0.930284    ┆ 0.001275 ┆ 0.984561    ┆ 0.91988

In [17]:
ksc = ksc.to_pandas()
ksc['Test Type'] = 'Real'
ksc_synth = ksc_synth.to_pandas()
ksc_synth['Test Type'] = 'Synth'

combined = pd.concat([ksc, ksc_synth])

In [18]:
pivot = combined.pivot(index="Algorithm", columns="Test Type")
pivot

Accuracy           Weighted Accuracy                Time  \
Test Type        Real     Synth              Real     Synth      Real   
Algorithm                                                               
CHI          0.003013  0.000872          0.004436  0.001404  0.075777   
CLASSIFIER   0.857003  0.790482          0.800860  0.726319  0.120951   
DC           0.839487  0.856540          0.784377  0.806411  0.010604   
FID          0.945962  0.861703          0.919281  0.810074  0.001913   
IRPR         0.735493  0.736819          0.688856  0.682549  0.894781   
MAUVE        0.960912  0.833163          0.938161  0.775197  0.001275   
PR           0.676019  0.745286          0.630614  0.687186  0.010606   
TRADITIONAL  0.923040  0.866206          0.890403  0.815964  0.000276   
ZERO         0.947468  0.859017          0.921543  0.811942  0.000026   
ZIPF         0.747523  0.837577          0.716545  0.799773  0.133122   

                      Monotonicity           Separability           Linearity  \
Test Type       Synth         Real     Synth         Real     Synth      Real   
Algorithm                                                                       
CHI          0.090069     0.109846  0.134012     0.716889  0.873021  0.111007   
CLASSIFIER   0.120917     0.921227  0.886135     0.724894  0.641392  0.920500   
DC           0.010739     0.913015  0.927103     0.849156  0.832722  0.927192   
FID          0.001970     0.930509  0.898860     0.809821  0.681367  0.944213   
IRPR         0.914143     0.762696  0.784939     0.438161  0.389587  0.787832   
MAUVE        0.001300     0.976853  0.917189     0.910648  0.790016  0.974217   
PR           0.010741     0.751213  0.813103     0.642376  0.607930  0.833554   
TRADITIONAL  0.000300     0.952817  0.933658     0.849525  0.769083  0.959559   
ZERO         0.000030     0.971030  0.917858     0.900664  0.739767  0.975605   
ZIPF         0.155006     0.732763  0.865222     0.267881  0.585600  0.713124   

                        Overall            
Test Type       Synth      Real     Synth  
Algorithm                                  
CHI          0.141436  0.166070  0.206802  
CLASSIFIER   0.896279  0.724239  0.676921  
DC           0.946736  0.720638  0.730042  
FID          0.912584  0.758616  0.694426  
IRPR         0.804098  0.717970  0.718689  
MAUVE        0.925602  0.793678  0.707078  
PR           0.853342  0.590730  0.619598  
TRADITIONAL  0.942996  0.762603  0.721368  
ZERO         0.928302  0.786056  0.709486  
ZIPF         0.870550  0.551826  0.685621

In [19]:
averages = {}
for column in pivot.columns:
    if column[0] in averages:
        for metric, value in zip(pivot[column].index, pivot[column]):
            averages[column[0]][metric].append(value)
        
    else:
        averages[column[0]] = {}
        for metric, value in zip(pivot[column].index, pivot[column]):
            averages[column[0]][metric] = [value]

final = {}
for metric, values in averages.items():
    final[metric] = {}
    for algorithm, result in values.items():
        final[metric][algorithm] = (sum(result))/(len(result))
        
pd.DataFrame(final).sort_values(by='Overall', ascending=False)

,Accuracy,Weighted Accuracy,Time,Monotonicity,Separability,Linearity,Overall
MAUVE,0.897037,0.856679,0.001288,0.947021,0.850332,0.949910,0.750378
ZERO,0.903242,0.866743,0.000028,0.944444,0.820215,0.951953,0.747771
TRADITIONAL,0.894623,0.853184,0.000288,0.943238,0.809304,0.951278,0.741986
FID,0.903832,0.864677,0.001942,0.914684,0.745594,0.928399,0.726521
DC,0.848013,0.795394,0.010672,0.920059,0.840939,0.936964,0.725340
IRPR,0.736156,0.685702,0.904462,0.773817,0.413874,0.795965,0.718329
CLASSIFIER,0.823742,0.763589,0.120934,0.903681,0.683143,0.908390,0.700580
ZIPF,0.792550,0.758159,0.144064,0.798992,0.426740,0.791837,0.618724
PR,0.710652,0.658900,0.010674,0.782158,0.625153,0.843448,0.605164
CHI,0.001942,0.002920,0.082923,0.121929,0.794955,0.126221,0.186436
